In [ ]:
# CODE FOR PLOT GENERATED WITH CLAUDE -------->

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

one_hot_array = np.array(one_hot_rows)
species_counts_array = one_hot_array.sum(axis=0)

# Sort by count descending for better readability
sorted_indices = np.argsort(species_counts_array)[::-1]
sorted_counts = species_counts_array[sorted_indices]
sorted_labels = [species_list[i] for i in sorted_indices]

# Botanical color palette — earthy greens, warm neutrals, soft teals
colors = [
    "#4A7C59", "#8DB87A", "#C9DDB5",
    "#6B9E87", "#A3C4A8", "#D4E8D1",
    "#7AABB8", "#3D6B72", "#B5D5DA",
    "#9EBF8E", "#5C8A6E", "#E2EED9",
]
colors = (colors * ((len(sorted_labels) // len(colors)) + 1))[:len(sorted_labels)]

fig, (ax_pie, ax_legend) = plt.subplots(
    1, 2,
    figsize=(14, 8)
)
fig.patch.set_facecolor("#F7F5F0")

# ── Pie chart ──────────────────────────────────────────────────────────────
wedges, texts, autotexts = ax_pie.pie(
    sorted_counts,
    labels=None,               # labels go in legend panel
    autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
    pctdistance=0.78,
    colors=colors,
    startangle=140,
    wedgeprops=dict(linewidth=1.2, edgecolor="#F7F5F0"),
    shadow=False,
)

for at in autotexts:
    at.set_fontsize(10)
    at.set_color("#2C2C2C")
    at.set_fontweight("semibold")

ax_pie.set_facecolor("#F7F5F0")
ax_pie.set_aspect('equal')
ax_pie.set_title(
    "Species Distribution",
    fontsize=17, fontweight="bold",
    color="#2C2C2C", pad=20,
    fontfamily="serif"
)

# ── Legend panel ───────────────────────────────────────────────────────────
ax_legend.set_facecolor("#F7F5F0")
ax_legend.axis("off")

total = sorted_counts.sum()
y = 0.98
row_h = min(0.072, 0.96 / len(sorted_labels))   # compress if many species

for i, (label, count, color) in enumerate(zip(sorted_labels, sorted_counts, colors)):
    pct = count / total * 100

    # Color swatch
    swatch = FancyBboxPatch(
        (0.0, y - 0.030), 0.055, 0.042,
        boxstyle="round,pad=0.003",
        facecolor=color, edgecolor="#F7F5F0", linewidth=0.8,
        transform=ax_legend.transAxes, clip_on=False
    )
    ax_legend.add_patch(swatch)

    # Species name (italic)
    ax_legend.text(
        0.085, y - 0.008,
        label.replace("_", " ").capitalize(),
        transform=ax_legend.transAxes,
        fontsize=15, color="#2C2C2C",
        va="center", fontstyle="italic"
    )

    # Count + percentage (right-aligned)
    ax_legend.text(
        0.98, y - 0.008,
        f"{int(count):,}  ({pct:.1f}%)",
        transform=ax_legend.transAxes,
        fontsize=15, color="#555555",
        va="center", ha="right"
    )

    y -= row_h

# Subtle divider line between pie and legend
fig.add_artist(
    plt.Line2D([0.615, 0.615], [0.08, 0.92],
               transform=fig.transFigure,
               color="#CCCCCC", linewidth=0.8)
)

# Sample count footnote
fig.text(
    0.5, 0.02,
    f"n = {int(total):,} samples  ·  {len(sorted_labels)} species",
    ha="center", fontsize=15, color="#888888", fontstyle="italic"
)

plt.subplots_adjust(left=0.02, right=0.97, top=0.93, bottom=0.07, wspace=0.05)
plt.savefig("species_distribution.png", dpi=180, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Compute per-row band means ---
# For each band, average across all pixels (441 values) for that row
band_means = pd.DataFrame({
    b: df[[c for c in df.columns if c.startswith(f"{b}_")]].mean(axis=1)
    for b in b_names
})
# band_means shape: (n_samples, 12)

# --- Get species labels from one-hot ---
species_labels = one_hot_df.idxmax(axis=1)  # Series of species name per row

# --- Compute mean and std per species per band ---
band_means["species"] = species_labels

stats = band_means.groupby("species")[b_names].agg(["mean", "std"])
# stats is MultiIndex columns: (band_name, "mean"/"std")

# --- Plot ---
species_list_present = stats.index.tolist()
n_species = len(species_list_present)
band_indices = np.arange(1, 13)

colors = cm.tab20(np.linspace(0, 1, n_species))

fig, ax = plt.subplots(figsize=(14, 6))

for i, species in enumerate(species_list_present):
    means = stats.loc[species, (b_names, "mean")].values
    stds  = stats.loc[species, (b_names, "std")].values

    ax.plot(band_indices, means, label=species, color=colors[i], linewidth=1.8)
    ax.fill_between(band_indices,
                    means - stds,
                    means + stds,
                    color=colors[i], alpha=0.15)

ax.set_xlabel("Band", fontsize=12)
ax.set_ylabel("Mean Reflectance (± std)", fontsize=12)
ax.set_title("Average Reflectance per Band per Species", fontsize=14)
ax.set_xticks(band_indices)
ax.set_xticklabels([f"Band {i}" for i in band_indices], rotation=45, ha="right")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, framealpha=0.7)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("spectral_signatures.png", dpi=150, bbox_inches="tight")
plt.show()